## Chat myPDF: RAG with LLAMA, LangChain, Ollama, and FAISS Vector Store
2. Retrievals and Augmented Data Generation (RAG)
- RAG PDF Dataset: https://github.com/laxmimerit/rag-dataset

In [ ]:
from langchain_ollama import OllamaEmbeddings

import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

In [ ]:
embeddings = OllamaEmbeddings(model='all-minilm', base_url='http://localhost:11434')
db_name = r"../08. Vector Stores and Retrievals\helath_supplements"
vector_store = FAISS.load_local(db_name, embeddings, allow_dangerous_deserialization=True)

In [ ]:
### Retreival
question = "how to gain muscle mass?"
docs = vector_store.search(query=question, k=5, search_type="similarity")

In [ ]:
docs

In [ ]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':3})
retriever.invoke(question)

In [ ]:
question='how to lose weight?'
retriever.invoke(question)

In [ ]:
retriever = vector_store.as_retriever(search_type='similarity_score_threshold', search_kwargs={'k':3,'score_threshold':0.1})
retriever.invoke(question)

In [ ]:
retriever = vector_store.as_retriever(search_type='mmr',search_kwargs={'k':3,'fetch_k':20,'lambda_mult':1})
docs = retriever.invoke(question)
docs

## RAG with LLAMA on OLLAMA

In [ ]:
from langchain_ollama import ChatOllama 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough 
from langchain_core.prompts import ChatPromptTemplate

# in langchain v1, hub is removed. please use the prompt directly. 
# from langchain import hub

In [ ]:
# prompt = hub.pull("rlm/rag-prompt")
prompt = """You are an assistant for question-answering tasks. 
            Use the following pieces of retrieved context to answer the question. 
            If you don't know the answer, just say that you don't know. 
            Use three sentences maximum and keep the answer concise.
            
            Question: {question}
            Context: {context}
            
            Answer:"""

In [ ]:
prompt = ChatPromptTemplate.from_template(prompt)
# prompt

In [ ]:
llm = ChatOllama(model='qwen3', base_url='http://localhost:11434')
llm.invoke('hi')

In [ ]:
def format_docs(docs):
    return '\n\n'.join([doc.page_content for doc in docs])

context = format_docs(docs)
# print(context)

In [ ]:
rag_chain = (
    {"context": retriever|format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
question = "how to lose weight?"
response = rag_chain.invoke(question)

print(response)

In [ ]:
question = "how to gain muscle mass?"
response = rag_chain.invoke(question)

print(response)